# Lab — Object detection from geometry to set prediction

**Scenario.** An industrial assembly line must find every housing, fastener, and contamination region in a tray image. Factories A and B provide development data; Factory C remains held out to expose capture shift.

This is a credential-free, CPU-safe laboratory. We implement the core geometry and evaluation contracts, train a tiny anchor-free dense detector, inject source and small-object failures, compare duplicate policies, and construct a DETR-style Hungarian assignment. The procedural corpus and tiny model are teaching instruments—not production or safety evidence.

![Classification, localization, and detection have different output contracts.](assets/detection-output-contract.svg)

## 0. Experiment contract

We will keep these boundaries explicit:

- boxes use half-open continuous `xyxy` coordinates internally;
- detector outputs are variable sets of `{boxes, labels, scores}`;
- Factory C is never used for gradient updates or threshold selection;
- the default run uses no network, credentials, remote code, or paid SDK;
- labels are used for supervised detection training and source-aware evaluation;
- AP is a transparent teaching implementation, not a substitute for official COCO evaluation;
- timing includes model forward, decoding, thresholding, and NMS in this notebook runtime; and
- every threshold is labeled as a demonstration choice, not a factory SLO.

The final artifact will distinguish locally measured evidence from optional TorchVision, Transformers, Ultralytics, and Grounding DINO paths.

In [ ]:
from __future__ import annotations

import importlib.metadata
import json
import math
import os
import platform
import random
import time
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from PIL import Image, ImageDraw, ImageEnhance
from scipy.optimize import linear_sum_assignment
from torch.utils.data import DataLoader, Dataset
from torchvision.ops import (
    box_iou as torchvision_box_iou,
    generalized_box_iou,
    generalized_box_iou_loss,
    nms as torchvision_nms,
    sigmoid_focal_loss,
)
from torchvision.transforms.functional import pil_to_tensor
from torchvision.utils import draw_bounding_boxes

SEED = 23
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(max(1, min(4, os.cpu_count() or 1)))

CPU_SAFE = os.getenv("CV_FULL_RUN", "0") != "1"
if CPU_SAFE:
    DEVICE = torch.device("cpu")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

ARTIFACT_DIR = Path(".artifacts/object_detection")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
IMAGE_SIZE = 128
GRID_SIZE = 8
CLASS_NAMES = ["housing", "fastener", "contamination"]
CLASS_COLORS = ["#2F6BFF", "#16A3A5", "#F59E42"]
DEMONSTRATION_THRESHOLD_NOTICE = "Demonstration thresholds for this procedural notebook corpus only."

environment = {
    "python": platform.python_version(),
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "scipy": scipy.__version__,
    "device": str(DEVICE),
    "cpu_safe": CPU_SAFE,
    "seed": SEED,
}
print(json.dumps(environment, indent=2))

## 1. Box representations are executable contracts

Internally we use pixel `xyxy = (x_min, y_min, x_max, y_max)`. Center `xywh` is useful for grid heads and YOLO-style annotations. COCO's top-left `xywh` is a different convention despite sharing the same shorthand.

The conversion functions below accept an `N×4` tensor, validate shape, and delay rounding. Normalization divides x coordinates by image width and y coordinates by image height.

In [ ]:
def ensure_boxes(boxes) -> torch.Tensor:
    values = torch.as_tensor(boxes, dtype=torch.float32)
    if values.ndim == 1:
        values = values.unsqueeze(0)
    if values.ndim != 2 or values.shape[1] != 4:
        raise ValueError("boxes must have shape [N, 4]")
    return values


def xyxy_to_xywh(boxes) -> torch.Tensor:
    boxes = ensure_boxes(boxes)
    x1, y1, x2, y2 = boxes.unbind(dim=1)
    return torch.stack(((x1 + x2) / 2, (y1 + y2) / 2, x2 - x1, y2 - y1), dim=1)


def xywh_to_xyxy(boxes) -> torch.Tensor:
    boxes = ensure_boxes(boxes)
    cx, cy, width, height = boxes.unbind(dim=1)
    return torch.stack((cx - width / 2, cy - height / 2, cx + width / 2, cy + height / 2), dim=1)


def xyxy_to_coco_xywh(boxes) -> torch.Tensor:
    boxes = ensure_boxes(boxes)
    x1, y1, x2, y2 = boxes.unbind(dim=1)
    return torch.stack((x1, y1, x2 - x1, y2 - y1), dim=1)


def normalize_boxes(boxes, image_width: int, image_height: int) -> torch.Tensor:
    scale = torch.tensor([image_width, image_height, image_width, image_height], dtype=torch.float32)
    return ensure_boxes(boxes) / scale


def denormalize_boxes(boxes, image_width: int, image_height: int) -> torch.Tensor:
    scale = torch.tensor([image_width, image_height, image_width, image_height], dtype=torch.float32)
    return ensure_boxes(boxes) * scale


def clip_boxes_xyxy(boxes, image_width: int, image_height: int) -> torch.Tensor:
    values = ensure_boxes(boxes).clone()
    values[:, 0::2] = values[:, 0::2].clamp(0, image_width)
    values[:, 1::2] = values[:, 1::2].clamp(0, image_height)
    return values


def validate_boxes(boxes, image_width: int, image_height: int) -> None:
    values = ensure_boxes(boxes)
    if not torch.isfinite(values).all():
        raise ValueError("boxes contain non-finite values")
    if (values[:, 2] <= values[:, 0]).any() or (values[:, 3] <= values[:, 1]).any():
        raise ValueError("every box must have positive width and height")
    if (values < 0).any() or (values[:, [0, 2]] > image_width).any() or (values[:, [1, 3]] > image_height).any():
        raise ValueError("boxes leave the image bounds")


example_xyxy = torch.tensor([[10.0, 20.0, 50.0, 70.0], [0.0, 0.0, 128.0, 128.0]])
example_xywh = xyxy_to_xywh(example_xyxy)
torch.testing.assert_close(xywh_to_xyxy(example_xywh), example_xyxy)
normalized = normalize_boxes(example_xywh, IMAGE_SIZE, IMAGE_SIZE)
torch.testing.assert_close(denormalize_boxes(normalized, IMAGE_SIZE, IMAGE_SIZE), example_xywh)
validate_boxes(example_xyxy, IMAGE_SIZE, IMAGE_SIZE)

box_contract_examples = {
    "xyxy_pixels": example_xyxy.tolist(),
    "xywh_center_pixels": example_xywh.tolist(),
    "coco_xywh_top_left_pixels": xyxy_to_coco_xywh(example_xyxy).tolist(),
    "xywh_center_normalized": normalized.tolist(),
}
print(json.dumps(box_contract_examples, indent=2))

### Annotation formats

The same object can be serialized through several interfaces. The next cell makes the semantic difference visible. Production converters must also carry image/category IDs, taxonomy version, source, ignore/crowd policy, and provenance.

In [ ]:
annotation_example = {
    "voc_like": {"name": "fastener", "bndbox": {"xmin": 10, "ymin": 20, "xmax": 50, "ymax": 70}},
    "coco_like": {"image_id": 7, "category_id": 2, "bbox": [10, 20, 40, 50], "area": 2000, "iscrowd": 0},
    "yolo_like": "1 0.234375 0.351562 0.312500 0.390625",
}
print(json.dumps(annotation_example, indent=2))

## 2. Implement Intersection over Union

IoU appears in assignment, localization-loss families, NMS, and evaluation. The implementation must clamp negative intersection dimensions and handle zero-area unions safely.

In [ ]:
def box_area(boxes) -> torch.Tensor:
    boxes = ensure_boxes(boxes)
    return (boxes[:, 2] - boxes[:, 0]).clamp_min(0) * (boxes[:, 3] - boxes[:, 1]).clamp_min(0)


def box_iou(boxes_a, boxes_b) -> torch.Tensor:
    a, b = ensure_boxes(boxes_a), ensure_boxes(boxes_b)
    top_left = torch.maximum(a[:, None, :2], b[None, :, :2])
    bottom_right = torch.minimum(a[:, None, 2:], b[None, :, 2:])
    intersection_wh = (bottom_right - top_left).clamp_min(0)
    intersection = intersection_wh[..., 0] * intersection_wh[..., 1]
    union = box_area(a)[:, None] + box_area(b)[None, :] - intersection
    return torch.where(union > 0, intersection / union, torch.zeros_like(union))


iou_cases = {
    "perfect": (torch.tensor([[10, 10, 30, 30]]), torch.tensor([[10, 10, 30, 30]]), 1.0),
    "partial": (torch.tensor([[0, 0, 20, 20]]), torch.tensor([[10, 10, 30, 30]]), 100 / 700),
    "none": (torch.tensor([[0, 0, 10, 10]]), torch.tensor([[20, 20, 30, 30]]), 0.0),
    "contained": (torch.tensor([[0, 0, 20, 20]]), torch.tensor([[5, 5, 15, 15]]), 0.25),
    "edge_touching": (torch.tensor([[0, 0, 10, 10]]), torch.tensor([[10, 0, 20, 10]]), 0.0),
}
iou_rows = []
for name, (first, second, expected) in iou_cases.items():
    manual = box_iou(first, second).item()
    reference = torchvision_box_iou(first.float(), second.float()).item()
    assert math.isclose(manual, expected, rel_tol=1e-6, abs_tol=1e-6)
    assert math.isclose(manual, reference, rel_tol=1e-6, abs_tol=1e-6)
    iou_rows.append({"case": name, "manual_iou": manual, "torchvision_iou": reference})
pd.DataFrame(iou_rows)

## 3. Generate a source-aware multi-object corpus

Every image contains one to four objects, and no two object centers share an 8×8 target cell. Factory C changes background, illumination, and texture. Object labels and boxes remain exact because they are produced by the same deterministic generator as the pixels.

**Size contract for this 128×128 corpus:** small `< 1.25%`, medium `< 4%`, and large `≥ 4%` of image area. These are not COCO's pixel thresholds.

In [ ]:
SOURCE_BACKGROUNDS = {
    "Factory A": (34, 40, 48),
    "Factory B": (43, 42, 50),
    "Factory C": (62, 51, 43),
}


@dataclass(frozen=True)
class DetectionSample:
    sample_id: str
    image: Image.Image
    boxes: np.ndarray
    labels: np.ndarray
    source: str


def area_band(box, image_size=IMAGE_SIZE) -> str:
    x1, y1, x2, y2 = box
    fraction = max(0, x2 - x1) * max(0, y2 - y1) / (image_size * image_size)
    if fraction < 0.0125:
        return "small"
    if fraction < 0.04:
        return "medium"
    return "large"


def make_detection_sample(source: str, source_index: int, item: int) -> DetectionSample:
    sample_seed = SEED * 100_000 + source_index * 10_000 + item
    rng = np.random.default_rng(sample_seed)
    background = np.full((IMAGE_SIZE, IMAGE_SIZE, 3), SOURCE_BACKGROUNDS[source], dtype=np.float32)
    horizontal = np.linspace(-9, 10, IMAGE_SIZE, dtype=np.float32)[None, :, None]
    vertical = np.linspace(-5, 7, IMAGE_SIZE, dtype=np.float32)[:, None, None]
    if source == "Factory A":
        background += horizontal
    elif source == "Factory B":
        background += vertical
        background[::12] += 5
    else:
        background += 1.35 * horizontal + 0.8 * vertical
        background[:, ::10] -= 6
    background += rng.normal(0, 2.5 if source != "Factory C" else 4.0, background.shape)
    image = Image.fromarray(np.uint8(np.clip(background, 0, 255)), mode="RGB")
    draw = ImageDraw.Draw(image)

    count = int(rng.integers(1, 5))
    used_cells, boxes, labels = set(), [], []
    attempts = 0
    while len(boxes) < count and attempts < 200:
        attempts += 1
        label = int(rng.choice([0, 1, 2], p=[0.38, 0.38, 0.24]))
        if label == 0:
            width, height = int(rng.integers(28, 45)), int(rng.integers(22, 37))
        elif label == 1:
            width = height = int(rng.integers(8, 15))
        else:
            width, height = int(rng.integers(12, 24)), int(rng.integers(10, 22))
        cx = int(rng.integers(width // 2 + 3, IMAGE_SIZE - width // 2 - 3))
        cy = int(rng.integers(height // 2 + 3, IMAGE_SIZE - height // 2 - 3))
        cell = (min(GRID_SIZE - 1, cy * GRID_SIZE // IMAGE_SIZE), min(GRID_SIZE - 1, cx * GRID_SIZE // IMAGE_SIZE))
        candidate = np.array([cx - width / 2, cy - height / 2, cx + width / 2, cy + height / 2], dtype=np.float32)
        if cell in used_cells:
            continue
        if boxes and box_iou(torch.tensor(candidate), torch.tensor(np.stack(boxes))).max().item() > 0.08:
            continue
        used_cells.add(cell)
        boxes.append(candidate)
        labels.append(label)
        coordinates = tuple(int(round(value)) for value in candidate)
        if label == 0:
            draw.rounded_rectangle(coordinates, radius=4, fill=(135, 154, 176), outline=(218, 229, 238), width=2)
        elif label == 1:
            draw.ellipse(coordinates, fill=(63, 177, 166), outline=(197, 244, 236), width=2)
            inner = (coordinates[0] + 3, coordinates[1] + 3, coordinates[2] - 3, coordinates[3] - 3)
            draw.ellipse(inner, fill=(40, 58, 65))
        else:
            contamination = (202, 91, 60) if item % 2 else (206, 157, 60)
            draw.ellipse(coordinates, fill=contamination, outline=(252, 213, 150), width=1)

    assert len(boxes) == count
    return DetectionSample(
        sample_id=f"{source_index}-{item:04d}",
        image=image,
        boxes=np.stack(boxes).astype(np.float32),
        labels=np.array(labels, dtype=np.int64),
        source=source,
    )


samples = []
counts = {"Factory A": 70, "Factory B": 70, "Factory C": 50}
for source_index, (source, count) in enumerate(counts.items()):
    samples.extend(make_detection_sample(source, source_index, item) for item in range(count))

train_samples = [sample for sample in samples if sample.source in {"Factory A", "Factory B"}]
test_samples = [sample for sample in samples if sample.source == "Factory C"]
object_rows = []
for sample in samples:
    for box, label in zip(sample.boxes, sample.labels):
        object_rows.append({"sample_id": sample.sample_id, "source": sample.source, "class_name": CLASS_NAMES[label], "size": area_band(box)})
object_metadata = pd.DataFrame(object_rows)
print(object_metadata.groupby(["source", "class_name", "size"]).size().rename("objects").to_frame())

In [ ]:
def render_sample(sample: DetectionSample, title: str | None = None):
    tensor = pil_to_tensor(sample.image)
    labels = [f"{CLASS_NAMES[label]} · {area_band(box)}" for box, label in zip(sample.boxes, sample.labels)]
    drawn = draw_bounding_boxes(
        tensor,
        boxes=torch.tensor(sample.boxes),
        labels=labels,
        colors=[CLASS_COLORS[label] for label in sample.labels],
        width=2,
    )
    plt.imshow(drawn.permute(1, 2, 0))
    plt.title(title or f"{sample.source} · {len(sample.boxes)} objects")
    plt.axis("off")


fig = plt.figure(figsize=(12, 4))
for index, source in enumerate(SOURCE_BACKGROUNDS, start=1):
    plt.subplot(1, 3, index)
    render_sample(next(sample for sample in samples if sample.source == source and len(sample.boxes) >= 3))
plt.tight_layout(); plt.show()

## 4. Detection matching and Average Precision

For each class, predictions are sorted globally by confidence. Within an image, the highest-IoU ground truth is matched only once. The implementation exposes every TP/FP event so duplicate, class, and localization errors can be inspected.

In [ ]:
def detection_events(predictions, targets, class_id: int, iou_threshold: float = 0.5, source: str | None = None, size: str | None = None):
    target_by_id = {target.sample_id: target for target in targets if source is None or target.source == source}
    matched = {}
    total_ground_truth = 0
    for sample_id, target in target_by_id.items():
        mask = target.labels == class_id
        if size is not None:
            mask &= np.array([area_band(box) == size for box in target.boxes])
        selected_boxes = target.boxes[mask]
        matched[sample_id] = np.zeros(len(selected_boxes), dtype=bool)
        total_ground_truth += len(selected_boxes)

    ranked = []
    for prediction in predictions:
        sample_id = prediction["sample_id"]
        if sample_id not in target_by_id:
            continue
        for box, label, score in zip(prediction["boxes"], prediction["labels"], prediction["scores"]):
            if int(label) != class_id:
                continue
            if size is not None and area_band(box) != size:
                continue
            ranked.append((float(score), sample_id, np.asarray(box, dtype=np.float32)))
    ranked.sort(key=lambda row: row[0], reverse=True)

    rows = []
    for score, sample_id, predicted_box in ranked:
        target = target_by_id[sample_id]
        mask = target.labels == class_id
        if size is not None:
            mask &= np.array([area_band(box) == size for box in target.boxes])
        candidate_boxes = target.boxes[mask]
        best_iou, best_index = 0.0, -1
        if len(candidate_boxes):
            overlaps = box_iou(torch.tensor(predicted_box), torch.tensor(candidate_boxes))[0].numpy()
            best_index = int(np.argmax(overlaps))
            best_iou = float(overlaps[best_index])
        is_true_positive = best_iou >= iou_threshold and best_index >= 0 and not matched[sample_id][best_index]
        duplicate = best_iou >= iou_threshold and best_index >= 0 and matched[sample_id][best_index]
        if is_true_positive:
            matched[sample_id][best_index] = True
        rows.append({
            "score": score,
            "sample_id": sample_id,
            "tp": int(is_true_positive),
            "fp": int(not is_true_positive),
            "duplicate": int(duplicate),
            "best_iou": best_iou,
        })
    return pd.DataFrame(rows), total_ground_truth


def precision_recall_ap(events: pd.DataFrame, total_ground_truth: int):
    if total_ground_truth == 0:
        return np.array([]), np.array([]), float("nan")
    if events.empty:
        return np.array([0.0]), np.array([0.0]), 0.0
    cumulative_tp = events.tp.to_numpy().cumsum()
    cumulative_fp = events.fp.to_numpy().cumsum()
    recall = cumulative_tp / total_ground_truth
    precision = cumulative_tp / np.maximum(cumulative_tp + cumulative_fp, 1)
    extended_recall = np.concatenate(([0.0], recall, [1.0]))
    extended_precision = np.concatenate(([0.0], precision, [0.0]))
    for index in range(len(extended_precision) - 2, -1, -1):
        extended_precision[index] = max(extended_precision[index], extended_precision[index + 1])
    changes = np.where(extended_recall[1:] != extended_recall[:-1])[0]
    ap = float(np.sum((extended_recall[changes + 1] - extended_recall[changes]) * extended_precision[changes + 1]))
    return precision, recall, ap


def evaluate_map(predictions, targets, iou_thresholds=(0.5,), source=None, size=None):
    rows = []
    for threshold in iou_thresholds:
        for class_id, class_name in enumerate(CLASS_NAMES):
            events, total_gt = detection_events(predictions, targets, class_id, threshold, source=source, size=size)
            precision, recall, ap = precision_recall_ap(events, total_gt)
            rows.append({
                "iou_threshold": float(threshold),
                "class_name": class_name,
                "source": source or "all",
                "size": size or "all",
                "ground_truth": total_gt,
                "predictions": len(events),
                "ap": ap,
                "final_precision": float(precision[-1]) if len(precision) else 0.0,
                "final_recall": float(recall[-1]) if len(recall) else 0.0,
                "duplicates": int(events.duplicate.sum()) if not events.empty else 0,
            })
    return pd.DataFrame(rows)


def simulated_predictions(targets, seed=SEED):
    rng = np.random.default_rng(seed)
    result = []
    for target in targets:
        boxes, labels, scores = [], [], []
        for gt_box, label in zip(target.boxes, target.labels):
            small = area_band(gt_box) == "small"
            miss_probability = 0.18 if small else 0.05
            if target.source == "Factory C":
                miss_probability += 0.10
            if rng.random() < miss_probability:
                continue
            jitter_scale = 2.8 if small else 3.8
            jitter = rng.normal(0, jitter_scale, 4)
            predicted = np.asarray(gt_box) + jitter
            predicted = clip_boxes_xyxy(predicted, IMAGE_SIZE, IMAGE_SIZE)[0].numpy()
            if predicted[2] <= predicted[0] + 1 or predicted[3] <= predicted[1] + 1:
                continue
            boxes.append(predicted); labels.append(int(label)); scores.append(float(rng.uniform(0.60, 0.96)))
            if rng.random() < 0.45:
                duplicate = clip_boxes_xyxy(predicted + rng.normal(0, 2.2, 4), IMAGE_SIZE, IMAGE_SIZE)[0].numpy()
                if duplicate[2] > duplicate[0] + 1 and duplicate[3] > duplicate[1] + 1:
                    boxes.append(duplicate); labels.append(int(label)); scores.append(float(rng.uniform(0.35, 0.82)))
        if rng.random() < 0.45:
            x1, y1 = rng.uniform(0, 96, 2)
            width, height = rng.uniform(8, 28, 2)
            boxes.append(np.array([x1, y1, min(128, x1 + width), min(128, y1 + height)], dtype=np.float32))
            labels.append(int(rng.integers(0, len(CLASS_NAMES)))); scores.append(float(rng.uniform(0.15, 0.65)))
        result.append({
            "sample_id": target.sample_id,
            "boxes": np.asarray(boxes, dtype=np.float32).reshape(-1, 4),
            "labels": np.asarray(labels, dtype=np.int64),
            "scores": np.asarray(scores, dtype=np.float32),
        })
    return result


evaluator_fixture = simulated_predictions(test_samples)
multi_iou = np.arange(0.50, 0.96, 0.05)
fixture_evaluation = evaluate_map(evaluator_fixture, test_samples, multi_iou)
fixture_summary = fixture_evaluation.groupby("iou_threshold").ap.mean().rename("mAP").reset_index()
fixture_summary.round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for class_id, class_name in enumerate(CLASS_NAMES):
    events, total_gt = detection_events(evaluator_fixture, test_samples, class_id, iou_threshold=0.5)
    precision, recall, ap = precision_recall_ap(events, total_gt)
    axes[0].plot(recall, precision, marker=".", label=f"{class_name} AP={ap:.2f}")
axes[0].set(xlabel="recall", ylabel="precision", title="Confidence-ranked PR curves at IoU 0.50", xlim=(0, 1.02), ylim=(0, 1.02))
axes[0].grid(alpha=0.25); axes[0].legend(fontsize=8)
axes[1].plot(fixture_summary.iou_threshold, fixture_summary.mAP, marker="o")
axes[1].set(xlabel="IoU threshold", ylabel="mean AP", title="Tighter localization criteria reduce AP", ylim=(0, 1.02))
axes[1].grid(alpha=0.25)
plt.tight_layout(); plt.show()
print({"AP50": round(fixture_summary.iloc[0].mAP, 3), "AP50_95": round(fixture_summary.mAP.mean(), 3)})

### AP50 versus AP75: one box, two verdicts

A correct class label is not enough. The next controlled geometry example uses one prediction with IoU near 0.62. It passes the 0.50 rule and fails the 0.75 rule, even though both boxes visibly refer to the same object.

In [ ]:
ap_example_ground_truth = torch.tensor([[20, 20, 80, 80]], dtype=torch.float32)
ap_example_prediction = torch.tensor([[34, 20, 94, 80]], dtype=torch.float32)
ap_example_iou = float(box_iou(ap_example_prediction, ap_example_ground_truth).item())
ap50_ap75_example = pd.DataFrame([{
    "prediction": "A",
    "IoU": ap_example_iou,
    "correct_at_AP50": ap_example_iou >= 0.50,
    "correct_at_AP75": ap_example_iou >= 0.75,
}])
display(ap50_ap75_example.round(3))

fig, ax = plt.subplots(figsize=(5, 4))
ax.add_patch(plt.Rectangle((20, 20), 60, 60, fill=False, linewidth=3, edgecolor="#16A3A5", label="ground truth"))
ax.add_patch(plt.Rectangle((34, 20), 60, 60, fill=False, linewidth=3, linestyle="--", edgecolor="#F59E42", label="prediction A"))
ax.set(xlim=(0, 110), ylim=(100, 0), aspect="equal", title=f"IoU={ap_example_iou:.2f}: TP at 0.50, FP at 0.75")
ax.legend(loc="lower right"); ax.grid(alpha=0.15); plt.tight_layout(); plt.show()
assert 0.50 <= ap_example_iou < 0.75

## 5. Anchor coverage versus anchor-free responsibility

Anchors create reference boxes at every feature location. The best-anchor IoU measures geometric coverage before any regression. Anchor-free targets assign each object center to one grid cell and predict the normalized box directly.

![Anchor and anchor-free paths use different responsibility contracts.](assets/anchor-vs-anchor-free.svg)

In [ ]:
def generate_anchors(grid_size: int, shapes, image_size: int = IMAGE_SIZE) -> torch.Tensor:
    stride = image_size / grid_size
    anchors = []
    for row in range(grid_size):
        for column in range(grid_size):
            cx, cy = (column + 0.5) * stride, (row + 0.5) * stride
            for width, height in shapes:
                anchors.append([cx - width / 2, cy - height / 2, cx + width / 2, cy + height / 2])
    return clip_boxes_xyxy(anchors, image_size, image_size)


anchor_banks = {
    "one square anchor": generate_anchors(GRID_SIZE, [(20, 20)]),
    "three shape anchors": generate_anchors(GRID_SIZE, [(10, 10), (22, 18), (38, 30)]),
}
coverage_rows = []
for bank_name, anchors in anchor_banks.items():
    for sample in train_samples:
        overlaps = box_iou(torch.tensor(sample.boxes), anchors)
        for box, label, best in zip(sample.boxes, sample.labels, overlaps.max(dim=1).values):
            coverage_rows.append({
                "anchor_bank": bank_name,
                "class_name": CLASS_NAMES[label],
                "size": area_band(box),
                "best_anchor_iou": best.item(),
                "covered_at_0_5": float(best >= 0.5),
            })
anchor_coverage = pd.DataFrame(coverage_rows)
coverage_summary = anchor_coverage.groupby(["anchor_bank", "size"]).agg(
    mean_best_iou=("best_anchor_iou", "mean"),
    coverage_at_0_5=("covered_at_0_5", "mean"),
    objects=("best_anchor_iou", "size"),
).reset_index()
display(coverage_summary.round(3))

cell_responsibilities = []
for sample in train_samples:
    occupied = set()
    for box in sample.boxes:
        cx, cy, _, _ = xyxy_to_xywh(box)[0]
        cell = (min(GRID_SIZE - 1, int(cy / IMAGE_SIZE * GRID_SIZE)), min(GRID_SIZE - 1, int(cx / IMAGE_SIZE * GRID_SIZE)))
        cell_responsibilities.append({"sample_id": sample.sample_id, "cell": cell, "collision": cell in occupied})
        occupied.add(cell)
assert not any(row["collision"] for row in cell_responsibilities)
print({"anchor_free_targets": len(cell_responsibilities), "grid_collisions": 0, "remaining_design_question": "which cell/level should own ambiguous objects?"})

## 6. Train a tiny anchor-free dense detector

The model emits an 8×8 grid. Each cell predicts:

```text
objectness logit + normalized center xywh + three class logits
```

The responsible center cell receives one target. Objectness uses focal loss, positive cells use class cross-entropy, and box geometry uses L1 plus GIoU. This is intentionally smaller than a modern YOLO detector: it makes assignment, loss components, decoding, and post-processing inspectable.

We train two runs:

- **baseline:** only source A/B pixels;
- **photometric mitigation:** source A/B plus brightness, contrast, and color-gain variation intended to reduce Factory C sensitivity.

The mitigation is an experiment, not a guaranteed improvement.

In [ ]:
def image_to_tensor(image: Image.Image) -> torch.Tensor:
    return pil_to_tensor(image).float() / 255.0


def encode_target(sample: DetectionSample):
    objectness = torch.zeros((GRID_SIZE, GRID_SIZE), dtype=torch.float32)
    boxes = torch.zeros((GRID_SIZE, GRID_SIZE, 4), dtype=torch.float32)
    classes = torch.full((GRID_SIZE, GRID_SIZE), -1, dtype=torch.long)
    normalized_xywh = normalize_boxes(xyxy_to_xywh(sample.boxes), IMAGE_SIZE, IMAGE_SIZE)
    for box, label in zip(normalized_xywh, sample.labels):
        column = min(GRID_SIZE - 1, int(box[0] * GRID_SIZE))
        row = min(GRID_SIZE - 1, int(box[1] * GRID_SIZE))
        if objectness[row, column] == 1:
            raise RuntimeError("two objects were assigned to one cell")
        objectness[row, column] = 1
        # Center offsets are local to the responsible cell; width and height
        # remain normalized to the image. This gives the dense head an explicit
        # spatial prior instead of asking it to infer absolute coordinates.
        boxes[row, column] = torch.tensor([
            box[0] * GRID_SIZE - column,
            box[1] * GRID_SIZE - row,
            box[2],
            box[3],
        ])
        classes[row, column] = int(label)
    return {"objectness": objectness, "boxes": boxes, "classes": classes}


class DetectionDataset(Dataset):
    def __init__(self, sample_rows, photometric_augmentation=False):
        self.sample_rows = list(sample_rows)
        self.photometric_augmentation = photometric_augmentation

    def __len__(self):
        return len(self.sample_rows)

    def __getitem__(self, index):
        sample = self.sample_rows[index]
        image = image_to_tensor(sample.image)
        if self.photometric_augmentation:
            gain = 0.72 + 0.60 * torch.rand((3, 1, 1))
            bias = -0.08 + 0.16 * torch.rand((3, 1, 1))
            contrast = 0.75 + 0.50 * torch.rand(())
            mean = image.mean(dim=(1, 2), keepdim=True)
            image = ((image - mean) * contrast + mean) * gain + bias
            image = image.clamp(0, 1)
        return image, encode_target(sample)


class TinyAnchorFreeDetector(nn.Module):
    def __init__(self, classes=len(CLASS_NAMES)):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Conv2d(3, 16, 3, stride=2, padding=1), nn.GroupNorm(4, 16), nn.SiLU(),
            nn.Conv2d(16, 32, 3, stride=2, padding=1), nn.GroupNorm(8, 32), nn.SiLU(),
            nn.Conv2d(32, 48, 3, stride=2, padding=1), nn.GroupNorm(8, 48), nn.SiLU(),
            nn.Conv2d(48, 64, 3, stride=2, padding=1), nn.GroupNorm(8, 64), nn.SiLU(),
        )
        self.head = nn.Sequential(
            nn.Conv2d(64, 64, 3, padding=1), nn.SiLU(),
            nn.Conv2d(64, 1 + 4 + classes, 1),
        )

    def forward(self, images):
        output = self.head(self.backbone(images))
        return output.permute(0, 2, 3, 1)


def local_xywh_to_global(boxes: torch.Tensor) -> torch.Tensor:
    """Convert [cell-x, cell-y, image-w, image-h] into image-normalized xywh."""
    rows, columns = torch.meshgrid(
        torch.arange(GRID_SIZE, device=boxes.device, dtype=boxes.dtype),
        torch.arange(GRID_SIZE, device=boxes.device, dtype=boxes.dtype),
        indexing="ij",
    )
    global_boxes = boxes.clone()
    global_boxes[..., 0] = (boxes[..., 0] + columns) / GRID_SIZE
    global_boxes[..., 1] = (boxes[..., 1] + rows) / GRID_SIZE
    return global_boxes


def detector_loss(raw_output, target):
    objectness_logits = raw_output[..., 0]
    predicted_boxes = raw_output[..., 1:5].sigmoid()
    class_logits = raw_output[..., 5:]
    target_objectness = target["objectness"].to(raw_output.device)
    target_boxes = target["boxes"].to(raw_output.device)
    target_classes = target["classes"].to(raw_output.device)
    positive = target_objectness.bool()
    positive_count = positive.sum().clamp_min(1)
    objectness_loss = sigmoid_focal_loss(
        objectness_logits,
        target_objectness,
        alpha=0.35,
        gamma=2.0,
        reduction="sum",
    ) / positive_count
    if positive.any():
        box_l1 = F.l1_loss(predicted_boxes[positive], target_boxes[positive])
        predicted_xyxy = xywh_to_xyxy(local_xywh_to_global(predicted_boxes)[positive])
        target_xyxy = xywh_to_xyxy(local_xywh_to_global(target_boxes)[positive])
        giou = generalized_box_iou_loss(predicted_xyxy, target_xyxy, reduction="mean")
        classification = F.cross_entropy(class_logits[positive], target_classes[positive])
    else:
        box_l1 = giou = classification = raw_output.sum() * 0
    total = 1.5 * objectness_loss + 5.0 * box_l1 + 2.0 * giou + classification
    return total, {
        "objectness": float(objectness_loss.detach()),
        "box_l1": float(box_l1.detach()),
        "giou": float(giou.detach()),
        "classification": float(classification.detach()),
    }


def train_detector(photometric_augmentation: bool, epochs: int):
    torch.manual_seed(SEED)
    model = TinyAnchorFreeDetector().to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)
    loader = DataLoader(
        DetectionDataset(train_samples, photometric_augmentation),
        batch_size=16,
        shuffle=True,
        num_workers=0,
        generator=torch.Generator().manual_seed(SEED),
    )
    history = []
    started = time.perf_counter()
    for epoch in range(epochs):
        model.train()
        epoch_rows = []
        for images, target in loader:
            images = images.to(DEVICE)
            raw = model(images)
            loss, components = detector_loss(raw, target)
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()
            epoch_rows.append({"loss": loss.item(), **components})
        means = pd.DataFrame(epoch_rows).mean().to_dict()
        history.append({"epoch": epoch + 1, "run": "photometric mitigation" if photometric_augmentation else "baseline", **means})
    elapsed = time.perf_counter() - started
    return model.cpu().eval(), pd.DataFrame(history), elapsed


TRAINING_EPOCHS = 30 if CPU_SAFE else 45
baseline_detector, baseline_history, baseline_seconds = train_detector(False, TRAINING_EPOCHS)
mitigated_detector, mitigated_history, mitigated_seconds = train_detector(True, TRAINING_EPOCHS)
training_history = pd.concat([baseline_history, mitigated_history], ignore_index=True)
print({
    "epochs": TRAINING_EPOCHS,
    "baseline_final_loss": round(baseline_history.loss.iloc[-1], 4),
    "mitigated_final_loss": round(mitigated_history.loss.iloc[-1], 4),
    "training_seconds": {"baseline": round(baseline_seconds, 1), "mitigated": round(mitigated_seconds, 1)},
})

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for run, group in training_history.groupby("run"):
    axes[0].plot(group.epoch, group.loss, marker="o", label=run)
axes[0].set(xlabel="epoch", ylabel="weighted training loss", title="Training objective")
axes[0].grid(alpha=0.25); axes[0].legend()
component_final = training_history.sort_values("epoch").groupby("run", as_index=False).tail(1)
component_final.set_index("run")[["objectness", "box_l1", "giou", "classification"]].T.plot.bar(ax=axes[1])
axes[1].set(ylabel="component value", title="Final raw loss components")
axes[1].tick_params(axis="x", rotation=25); axes[1].grid(axis="y", alpha=0.25)
plt.tight_layout(); plt.show()

## 7. Decode dense candidates and implement class-aware NMS

Every grid cell can emit a candidate. We multiply objectness by the maximum class probability, filter by confidence, convert normalized center boxes to pixel `xyxy`, and apply NMS independently per class.

![NMS greedily keeps the highest-scoring hypothesis and suppresses same-class overlap.](assets/nms-duplicate-removal.svg)

In [ ]:
def manual_nms(boxes, scores, iou_threshold: float) -> torch.Tensor:
    boxes, scores = ensure_boxes(boxes), torch.as_tensor(scores, dtype=torch.float32)
    if len(boxes) == 0:
        return torch.empty((0,), dtype=torch.long)
    order = scores.argsort(descending=True)
    keep = []
    while len(order):
        current = int(order[0])
        keep.append(current)
        if len(order) == 1:
            break
        remaining = order[1:]
        overlaps = box_iou(boxes[current], boxes[remaining])[0]
        order = remaining[overlaps <= iou_threshold]
    return torch.tensor(keep, dtype=torch.long)


def class_aware_nms(boxes, scores, labels, iou_threshold: float) -> torch.Tensor:
    boxes = ensure_boxes(boxes)
    scores = torch.as_tensor(scores, dtype=torch.float32)
    labels = torch.as_tensor(labels, dtype=torch.long)
    keep = []
    for label in labels.unique(sorted=True):
        selected = torch.where(labels == label)[0]
        local_keep = manual_nms(boxes[selected], scores[selected], iou_threshold)
        keep.extend(selected[local_keep].tolist())
    return torch.tensor(sorted(keep, key=lambda index: float(scores[index]), reverse=True), dtype=torch.long)


def decode_output(raw_output, confidence_threshold=0.25, nms_iou=0.5):
    if raw_output.ndim == 4:
        raw_output = raw_output[0]
    objectness = raw_output[..., 0].sigmoid()
    boxes_xywh = local_xywh_to_global(raw_output[..., 1:5].sigmoid()).reshape(-1, 4)
    class_probabilities = raw_output[..., 5:].softmax(dim=-1).reshape(-1, len(CLASS_NAMES))
    class_scores, labels = class_probabilities.max(dim=1)
    scores = objectness.reshape(-1) * class_scores
    selected = scores >= confidence_threshold
    boxes = xywh_to_xyxy(boxes_xywh[selected])
    boxes = denormalize_boxes(boxes, IMAGE_SIZE, IMAGE_SIZE)
    boxes = clip_boxes_xyxy(boxes, IMAGE_SIZE, IMAGE_SIZE)
    valid = (boxes[:, 2] > boxes[:, 0] + 1) & (boxes[:, 3] > boxes[:, 1] + 1)
    boxes, scores, labels = boxes[valid], scores[selected][valid], labels[selected][valid]
    pre_nms = len(boxes)
    keep = class_aware_nms(boxes.cpu(), scores.cpu(), labels.cpu(), nms_iou)
    return {
        "boxes": boxes.cpu()[keep].numpy(),
        "scores": scores.cpu()[keep].numpy(),
        "labels": labels.cpu()[keep].numpy(),
        "pre_nms_candidates": pre_nms,
    }


nms_test_boxes = torch.tensor([[10, 10, 50, 50], [12, 12, 49, 49], [60, 60, 90, 90]], dtype=torch.float32)
nms_test_scores = torch.tensor([0.95, 0.85, 0.70])
manual_keep = manual_nms(nms_test_boxes, nms_test_scores, 0.5)
reference_keep = torchvision_nms(nms_test_boxes, nms_test_scores, 0.5)
torch.testing.assert_close(manual_keep, reference_keep)
print({"manual_keep": manual_keep.tolist(), "torchvision_keep": reference_keep.tolist()})

### NMS failure injection: two real objects, one suppressed

NMS does not know whether overlap means duplicate predictions or two legitimate crowded objects. These two same-class boxes represent different objects. Aggressive NMS at 0.50 suppresses one; a 0.70 threshold retains both. This is a policy trade-off, not proof that 0.70 is universally better.

In [ ]:
crowded_object_boxes = torch.tensor([[10, 10, 60, 60], [22, 10, 72, 60]], dtype=torch.float32)
crowded_object_scores = torch.tensor([0.95, 0.90])
crowded_iou = float(box_iou(crowded_object_boxes[0], crowded_object_boxes[1]).item())
nms_crowding_rows = []
for threshold in [0.50, 0.70]:
    kept = manual_nms(crowded_object_boxes, crowded_object_scores, threshold)
    nms_crowding_rows.append({
        "nms_iou": threshold,
        "legitimate_objects": 2,
        "kept_predictions": len(kept),
        "crowded_scene_recall": len(kept) / 2,
        "kept_indices": kept.tolist(),
    })
nms_crowding_example = pd.DataFrame(nms_crowding_rows)
display(nms_crowding_example)

canvas = torch.full((3, 84, 84), 245, dtype=torch.uint8)
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for axis, threshold in zip(axes, [0.50, 0.70]):
    kept = manual_nms(crowded_object_boxes, crowded_object_scores, threshold)
    rendered = draw_bounding_boxes(
        canvas,
        crowded_object_boxes[kept],
        labels=[f"real object {index + 1}" for index in kept.tolist()],
        colors=["#16A3A5", "#F59E42"][: len(kept)],
        width=2,
    )
    axis.imshow(rendered.permute(1, 2, 0))
    axis.set_title(f"NMS {threshold:.2f}: kept {len(kept)}/2")
    axis.axis("off")
plt.suptitle(f"Two legitimate boxes overlap at IoU={crowded_iou:.2f}")
plt.tight_layout(); plt.show()
assert len(manual_nms(crowded_object_boxes, crowded_object_scores, 0.50)) == 1
assert len(manual_nms(crowded_object_boxes, crowded_object_scores, 0.70)) == 2

## 8. Evaluate source, size, confidence, duplicates, and latency

The same decoder contract is used for every run. We first compare A/B development images with held-out Factory C, then sweep confidence and NMS thresholds **only on a development subset**. Final Factory C reporting uses the selected development operating point.

In [ ]:
def predict_samples(model, sample_rows, confidence_threshold, nms_iou):
    model = model.eval().to(DEVICE)
    predictions, timings = [], []
    with torch.inference_mode():
        for sample in sample_rows:
            image = image_to_tensor(sample.image).unsqueeze(0).to(DEVICE)
            started = time.perf_counter()
            raw = model(image)
            decoded = decode_output(raw, confidence_threshold, nms_iou)
            timings.append((time.perf_counter() - started) * 1000)
            predictions.append({"sample_id": sample.sample_id, **decoded})
    model.cpu()
    return predictions, np.asarray(timings)


development_samples = train_samples[-40:]
model_runs = {
    "baseline": baseline_detector,
    "photometric mitigation": mitigated_detector,
}
default_predictions = {}
evaluation_rows = []
latency_rows = []
for run, model in model_runs.items():
    for split_name, sample_rows in {"development A/B": development_samples, "held-out Factory C": test_samples}.items():
        predictions, timings = predict_samples(model, sample_rows, confidence_threshold=0.20, nms_iou=0.5)
        default_predictions[(run, split_name)] = predictions
        evaluated = evaluate_map(predictions, sample_rows, np.arange(0.50, 0.96, 0.05))
        evaluation_rows.append({
            "run": run,
            "split": split_name,
            "AP50": evaluated[evaluated.iou_threshold.eq(0.5)].ap.mean(),
            "AP50_95": evaluated.ap.mean(),
            "mean_predictions_per_image": float(np.mean([len(item["boxes"]) for item in predictions])),
            "mean_pre_nms_per_image": float(np.mean([item["pre_nms_candidates"] for item in predictions])),
        })
        latency_rows.append({
            "run": run,
            "split": split_name,
            "median_ms": float(np.median(timings)),
            "p90_ms": float(np.percentile(timings, 90)),
            "iqr_ms": float(np.percentile(timings, 75) - np.percentile(timings, 25)),
        })
model_evaluation = pd.DataFrame(evaluation_rows)
latency_results = pd.DataFrame(latency_rows)
display(model_evaluation.round(3))
display(latency_results.round(2))

In [ ]:
threshold_rows = []
for run, model in model_runs.items():
    for confidence in [0.10, 0.20, 0.30, 0.40]:
        for nms_iou in [0.30, 0.50, 0.70]:
            predictions, _ = predict_samples(model, development_samples, confidence, nms_iou)
            evaluation = evaluate_map(predictions, development_samples, (0.5,))
            fastener = evaluation[evaluation.class_name.eq("fastener")].iloc[0]
            threshold_rows.append({
                "run": run,
                "confidence_threshold": confidence,
                "nms_iou": nms_iou,
                "mAP50": evaluation.ap.mean(),
                "macro_recall": evaluation.final_recall.mean(),
                "fastener_recall": fastener.final_recall,
                "duplicates": evaluation.duplicates.sum(),
                "mean_predictions_per_image": float(np.mean([len(item["boxes"]) for item in predictions])),
            })
threshold_sweep = pd.DataFrame(threshold_rows)
threshold_sweep["teaching_utility"] = (
    0.45 * threshold_sweep.mAP50
    + 0.45 * threshold_sweep.fastener_recall
    - 0.01 * threshold_sweep.mean_predictions_per_image
)
selected_operating_points = (
    threshold_sweep.sort_values("teaching_utility", ascending=False)
    .groupby("run", as_index=False)
    .head(1)
)
display(selected_operating_points.round(3))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for (run, nms_iou), group in threshold_sweep.groupby(["run", "nms_iou"]):
    axes[0].plot(group.confidence_threshold, group.fastener_recall, marker="o", label=f"{run}, NMS {nms_iou}")
axes[0].set(xlabel="confidence threshold", ylabel="fastener recall", title="Thresholds change the operating point", ylim=(0, 1.02))
axes[0].grid(alpha=0.25); axes[0].legend(fontsize=6)
for run, group in threshold_sweep.groupby("run"):
    axes[1].scatter(group.mean_predictions_per_image, group.mAP50, label=run, alpha=0.75)
axes[1].set(xlabel="mean detections / image", ylabel="development mAP50", title="Quality versus review volume")
axes[1].grid(alpha=0.25); axes[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
final_predictions = {}
final_eval_rows = []
source_size_rows = []
for run, model in model_runs.items():
    operating = selected_operating_points.set_index("run").loc[run]
    predictions, timings = predict_samples(
        model,
        test_samples,
        float(operating.confidence_threshold),
        float(operating.nms_iou),
    )
    final_predictions[run] = predictions
    evaluated = evaluate_map(predictions, test_samples, np.arange(0.50, 0.96, 0.05))
    final_eval_rows.append({
        "run": run,
        "AP50": evaluated[evaluated.iou_threshold.eq(0.5)].ap.mean(),
        "AP50_95": evaluated.ap.mean(),
        "confidence_threshold": float(operating.confidence_threshold),
        "nms_iou": float(operating.nms_iou),
        "median_ms": float(np.median(timings)),
        "p90_ms": float(np.percentile(timings, 90)),
    })
    for size in ["small", "medium", "large"]:
        sliced = evaluate_map(predictions, test_samples, (0.5,), size=size)
        source_size_rows.append({
            "run": run,
            "source": "Factory C",
            "size": size,
            "mAP50": sliced.ap.mean(),
            "macro_recall": sliced.final_recall.mean(),
            "ground_truth": sliced.ground_truth.sum(),
        })
final_evaluation = pd.DataFrame(final_eval_rows).sort_values("AP50", ascending=False)
source_size_evaluation = pd.DataFrame(source_size_rows)
display(final_evaluation.round(3))
display(source_size_evaluation.round(3))

### What photometric mitigation changed—and what it did not

The mitigation randomizes brightness, channel gain, bias, and contrast during training. It targets part of Factory C's capture change; it does **not** model every camera, texture, geometry, taxonomy, or process shift. We therefore compare held-out deltas and retain the residual development gap and size failures as evidence.

> A gain on this deterministic teaching corpus is evidence about this intervention under this generator—not a general claim that photometric augmentation solves source shift.

In [ ]:
def evaluation_value(run, split, metric):
    return float(model_evaluation.set_index(["run", "split"]).loc[(run, split), metric])


mitigation_rows = []
for metric in ["AP50", "AP50_95"]:
    baseline_value = evaluation_value("baseline", "held-out Factory C", metric)
    mitigated_value = evaluation_value("photometric mitigation", "held-out Factory C", metric)
    mitigation_rows.append({"metric": f"held-out {metric}", "baseline": baseline_value, "mitigated": mitigated_value, "delta": mitigated_value - baseline_value})
for size in ["small", "medium", "large"]:
    sliced = source_size_evaluation.set_index(["run", "size"])
    baseline_value = float(sliced.loc[("baseline", size), "mAP50"])
    mitigated_value = float(sliced.loc[("photometric mitigation", size), "mAP50"])
    mitigation_rows.append({"metric": f"held-out {size} mAP50", "baseline": baseline_value, "mitigated": mitigated_value, "delta": mitigated_value - baseline_value})
mitigation_comparison = pd.DataFrame(mitigation_rows)
display(mitigation_comparison.round(3))

mitigated_development_ap50 = evaluation_value("photometric mitigation", "development A/B", "AP50")
mitigated_heldout_ap50 = evaluation_value("photometric mitigation", "held-out Factory C", "AP50")
mitigation_limits = {
    "what_changed": "brightness, contrast, per-channel gain, and bias during training",
    "remaining_development_to_factory_c_AP50_gap": round(mitigated_development_ap50 - mitigated_heldout_ap50, 3),
    "remaining_small_object_mAP50": round(float(source_size_evaluation.set_index(["run", "size"]).loc[("photometric mitigation", "small"), "mAP50"]), 3),
    "what_still_failed": "strict localization, small/medium objects, and unmodeled source changes remain weaker",
    "generalization_warning": "photometric augmentation is one tested intervention, not a general source-shift solution",
}
print(json.dumps(mitigation_limits, indent=2))

In [ ]:
def draw_prediction(sample, prediction, score_digits=2):
    tensor = pil_to_tensor(sample.image)
    labels = [f"{CLASS_NAMES[int(label)]} {score:.{score_digits}f}" for label, score in zip(prediction["labels"], prediction["scores"])]
    if len(prediction["boxes"]):
        drawn = draw_bounding_boxes(
            tensor,
            torch.tensor(prediction["boxes"]),
            labels=labels,
            colors=[CLASS_COLORS[int(label)] for label in prediction["labels"]],
            width=2,
        )
    else:
        drawn = tensor
    return drawn.permute(1, 2, 0)


example_index = max(range(len(test_samples)), key=lambda index: len(test_samples[index].boxes))
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
plt.sca(axes[0]); render_sample(test_samples[example_index], "ground truth · Factory C")
for axis, run in zip(axes[1:], model_runs):
    axis.imshow(draw_prediction(test_samples[example_index], final_predictions[run][example_index]))
    axis.set_title(run); axis.axis("off")
plt.tight_layout(); plt.show()

### Confidence is not correctness probability

We define one prediction as correct when its class is correct, IoU is at least 0.50, and it uniquely matches a ground truth. Reliability bins are computed **after** thresholding and NMS, so the result is conditional on that selection policy.

In [ ]:
def all_class_event_table(predictions, targets, iou_threshold=0.5):
    rows = []
    for class_id, class_name in enumerate(CLASS_NAMES):
        events, _ = detection_events(predictions, targets, class_id, iou_threshold)
        if not events.empty:
            events = events.assign(class_name=class_name)
            rows.append(events)
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()


calibration_rows = []
for run, predictions in final_predictions.items():
    events = all_class_event_table(predictions, test_samples)
    if events.empty:
        continue
    events["confidence_bin"] = pd.cut(events.score, bins=np.linspace(0, 1, 6), include_lowest=True)
    grouped = events.groupby("confidence_bin", observed=False).agg(
        count=("tp", "size"),
        mean_confidence=("score", "mean"),
        empirical_precision=("tp", "mean"),
    ).reset_index()
    grouped["run"] = run
    grouped["absolute_gap"] = (grouped.mean_confidence - grouped.empirical_precision).abs()
    grouped["weighted_gap"] = grouped.absolute_gap * grouped["count"] / grouped["count"].sum()
    calibration_rows.append(grouped)
calibration_results = pd.concat(calibration_rows, ignore_index=True)
calibration_summary = calibration_results.groupby("run").weighted_gap.sum().rename("post_nms_ece").reset_index()
display(calibration_results)
display(calibration_summary.round(3))

## 9. Turn predictions into an automated error taxonomy

Aggregate AP tells us that a detector failed, not how. The taxonomy below logs every returned prediction as correct, localization error, wrong class, duplicate, or background false positive, then adds every unmatched target as a missed object. Rates are shares of logged events; one real object can create both a prediction error and a missed-target event.

Small-object and source-shift failures are **slices across these events**, not mutually exclusive instance labels.

![Detection failures are separated into per-instance outcomes and cross-instance size/source slices.](assets/detection-error-taxonomy.svg)

In [ ]:
ERROR_ORDER = ["Correct", "Missed object", "Localization error", "Wrong class", "Duplicate", "Background FP"]


def detection_error_events(predictions, targets, match_iou=0.50, localization_floor=0.10):
    target_by_id = {target.sample_id: target for target in targets}
    rows = []
    for prediction in predictions:
        sample_id = prediction["sample_id"]
        target = target_by_id[sample_id]
        matched_targets = set()
        order = np.argsort(-np.asarray(prediction["scores"]))
        for prediction_index in order:
            box = np.asarray(prediction["boxes"][prediction_index], dtype=np.float32)
            label = int(prediction["labels"][prediction_index])
            score = float(prediction["scores"][prediction_index])
            overlaps = box_iou(torch.tensor(box), torch.tensor(target.boxes))[0].numpy() if len(target.boxes) else np.array([])
            best_target = int(np.argmax(overlaps)) if len(overlaps) else -1
            best_iou = float(overlaps[best_target]) if best_target >= 0 else 0.0
            if best_iou >= match_iou and label == int(target.labels[best_target]):
                if best_target in matched_targets:
                    error_type = "Duplicate"
                else:
                    error_type = "Correct"
                    matched_targets.add(best_target)
            elif best_iou >= match_iou:
                error_type = "Wrong class"
            elif best_iou >= localization_floor:
                error_type = "Localization error"
            else:
                error_type = "Background FP"
            rows.append({
                "sample_id": sample_id,
                "event": "prediction",
                "error_type": error_type,
                "score": score,
                "best_iou": best_iou,
                "predicted_class": CLASS_NAMES[label],
                "target_class": CLASS_NAMES[int(target.labels[best_target])] if best_target >= 0 else None,
                "size": area_band(target.boxes[best_target]) if best_target >= 0 else area_band(box),
                "source": target.source,
            })
        for target_index, (target_box, target_label) in enumerate(zip(target.boxes, target.labels)):
            if target_index not in matched_targets:
                rows.append({
                    "sample_id": sample_id,
                    "event": "unmatched_target",
                    "error_type": "Missed object",
                    "score": np.nan,
                    "best_iou": np.nan,
                    "predicted_class": None,
                    "target_class": CLASS_NAMES[int(target_label)],
                    "size": area_band(target_box),
                    "source": target.source,
                })
    return pd.DataFrame(rows)


error_taxonomy_events = detection_error_events(final_predictions["photometric mitigation"], test_samples)
counts = error_taxonomy_events.groupby("error_type").size().reindex(ERROR_ORDER, fill_value=0)
examples = error_taxonomy_events.groupby("error_type").sample_id.agg(lambda ids: ", ".join(list(dict.fromkeys(ids))[:3])).reindex(ERROR_ORDER, fill_value="")
error_taxonomy_summary = pd.DataFrame({
    "Error type": ERROR_ORDER,
    "Count": counts.to_numpy(),
    "Rate": counts.to_numpy() / max(int(counts.sum()), 1),
    "Example IDs": examples.to_numpy(),
})
error_taxonomy_display = error_taxonomy_summary.copy()
error_taxonomy_display["Rate"] = error_taxonomy_display["Rate"].map(lambda value: f"{value:.1%}")
display(error_taxonomy_display)

failure_slices = pd.DataFrame([
    {
        "slice": "small-object miss rate",
        "value": float(
            len(error_taxonomy_events[
                error_taxonomy_events["size"].eq("small")
                & error_taxonomy_events["error_type"].eq("Missed object")
            ])
            / max(sum(area_band(box) == "small" for sample in test_samples for box in sample.boxes), 1)
        ),
        "interpretation": "unmatched small targets / all small ground truth",
    },
    {
        "slice": "development-to-Factory-C AP50 gap",
        "value": mitigated_development_ap50 - mitigated_heldout_ap50,
        "interpretation": "aggregate source-shift evidence",
    },
])
display(failure_slices.round({"value": 3}))
assert set(ERROR_ORDER) == set(error_taxonomy_summary["Error type"])
assert int(error_taxonomy_summary.Count.sum()) == len(error_taxonomy_events)

## 10. DETR-style set prediction and Hungarian matching

Dense detectors assign many locations and later resolve duplicates. DETR-style systems allocate learned query slots and train a global one-to-one assignment.

![Dense and set-prediction detectors handle candidate responsibility and duplicates differently.](assets/dense-vs-set-prediction.svg)

The next diagram zooms into the matching step: learned queries produce a prediction set; predictions and labeled boxes jointly define the cost matrix; the Hungarian algorithm chooses the globally minimum one-to-one assignment.

![Object queries and ground truth meet in a weighted cost matrix before Hungarian one-to-one assignment.](assets/detr-matching.svg)

The next cell uses five synthetic queries for three objects. It exposes the classification, normalized L1, and GIoU contributions separately. Changing their weights can change the globally optimal assignment; matching is not simply “choose the nearest box.”

In [ ]:
matching_sample = next(sample for sample in test_samples if len(sample.boxes) == 3)
ground_truth_boxes = torch.tensor(matching_sample.boxes, dtype=torch.float32)
ground_truth_labels = torch.tensor(matching_sample.labels, dtype=torch.long)
rng = np.random.default_rng(SEED + 91)
query_boxes = [
    clip_boxes_xyxy(box + torch.tensor(rng.normal(0, 4, 4), dtype=torch.float32), IMAGE_SIZE, IMAGE_SIZE)[0]
    for box in ground_truth_boxes
]
query_boxes += [torch.tensor([4, 8, 30, 32], dtype=torch.float32), torch.tensor([86, 78, 121, 116], dtype=torch.float32)]
query_boxes = torch.stack(query_boxes)
query_class_logits = torch.full((5, len(CLASS_NAMES) + 1), -1.5)
for index, label in enumerate(ground_truth_labels):
    query_class_logits[index, label] = 2.4
query_class_logits[3, -1] = 2.0
query_class_logits[4, -1] = 1.8
class_probabilities = query_class_logits.softmax(dim=1)

classification_cost = -class_probabilities[:, ground_truth_labels]
normalized_queries = normalize_boxes(query_boxes, IMAGE_SIZE, IMAGE_SIZE)
normalized_targets = normalize_boxes(ground_truth_boxes, IMAGE_SIZE, IMAGE_SIZE)
l1_cost = torch.cdist(normalized_queries, normalized_targets, p=1)
giou_cost = 1 - generalized_box_iou(query_boxes, ground_truth_boxes)
MATCH_WEIGHTS = {"classification": 1.0, "l1": 2.0, "giou": 2.0}
weighted_classification = MATCH_WEIGHTS["classification"] * classification_cost
weighted_l1 = MATCH_WEIGHTS["l1"] * l1_cost
weighted_giou = MATCH_WEIGHTS["giou"] * giou_cost
hungarian_cost = weighted_classification + weighted_l1 + weighted_giou
query_indices, target_indices = linear_sum_assignment(hungarian_cost.detach().numpy())
assigned_pairs = set(zip(query_indices.tolist(), target_indices.tolist()))

decomposition_rows = []
for query_index in range(len(query_boxes)):
    for target_index in range(len(ground_truth_boxes)):
        decomposition_rows.append({
            "query": query_index,
            "ground_truth": target_index,
            "classification_cost": float(classification_cost[query_index, target_index]),
            "L1_box_cost": float(l1_cost[query_index, target_index]),
            "GIoU_cost": float(giou_cost[query_index, target_index]),
            "weighted_classification": float(weighted_classification[query_index, target_index]),
            "weighted_L1": float(weighted_l1[query_index, target_index]),
            "weighted_GIoU": float(weighted_giou[query_index, target_index]),
            "total_matching_cost": float(hungarian_cost[query_index, target_index]),
            "assigned": (query_index, target_index) in assigned_pairs,
        })
hungarian_cost_decomposition = pd.DataFrame(decomposition_rows)
display(hungarian_cost_decomposition[hungarian_cost_decomposition.assigned].round(3))

hungarian_assignment = pd.DataFrame({
    "query": query_indices,
    "ground_truth": target_indices,
    "class": [CLASS_NAMES[int(ground_truth_labels[index])] for index in target_indices],
    "cost": hungarian_cost[query_indices, target_indices].detach().numpy(),
})

cost_panels = [
    (classification_cost, "classification cost"),
    (weighted_l1, "2 × L1 box cost"),
    (weighted_giou, "2 × GIoU cost"),
    (hungarian_cost, "total matching cost"),
]
fig, axes = plt.subplots(1, 4, figsize=(15, 4), sharex=True, sharey=True)
for axis, (matrix, title) in zip(axes, cost_panels):
    values = matrix.detach().numpy()
    image = axis.imshow(values, cmap="magma_r")
    for query_index, target_index in zip(query_indices, target_indices):
        axis.scatter(target_index, query_index, s=180, facecolors="none", edgecolors="#16A3A5", linewidths=2)
    axis.set(title=title, xlabel="ground truth", xticks=range(len(ground_truth_boxes)), yticks=range(len(query_boxes)))
    plt.colorbar(image, ax=axis, fraction=0.046, pad=0.04)
axes[0].set_ylabel("query")
plt.suptitle("Matching cost decomposition · teal circles show the global assignment")
plt.tight_layout(); plt.show()
assert np.allclose(
    hungarian_cost.detach().numpy(),
    (weighted_classification + weighted_l1 + weighted_giou).detach().numpy(),
)

**Interpretation.** Every ground-truth object receives exactly one query in the minimum-cost assignment. Remaining queries learn `no object`. One-to-one responsibility is the conceptual reason standard DETR inference has no traditional NMS; it is not a promise that similar predictions can never appear.

The DINO detector extends DETR-style training. It is unrelated to Course 04's DINO self-supervised representation learner beyond sharing an acronym.

## 11. Maintained SDK paths—guarded and excluded from local evidence

The learning implementation uses common PyTorch, torchvision, and SciPy APIs. Production candidates are opt-in because they download checkpoints, add licenses, and can execute repository/model code.

| Path | Environment flag | Governance gate |
| --- | --- | --- |
| TorchVision Faster R-CNN MobileNetV3 FPN | `CV_ENABLE_TORCHVISION_DETECTOR=1` | official weight enum, preprocessing, download/cache, target latency |
| Ultralytics YOLO26 | `CV_ENABLE_ULTRALYTICS=1` | exact `ultralytics==8.4.138`, AGPL-3.0 or enterprise license, `yolo26n.pt` provenance |
| Transformers Grounding DINO | `CV_ENABLE_GROUNDING_DINO=1` | exact model revision required, model/license card, prompt contract, artifact digest |

These paths are adapter examples, not benchmark results in this notebook.

In [ ]:
sdk_summary = []

ENABLE_TORCHVISION_DETECTOR = os.getenv("CV_ENABLE_TORCHVISION_DETECTOR", "0") == "1"
if ENABLE_TORCHVISION_DETECTOR:
    from torchvision.models.detection import (
        FasterRCNN_MobileNet_V3_Large_320_FPN_Weights,
        fasterrcnn_mobilenet_v3_large_320_fpn,
    )
    weights = FasterRCNN_MobileNet_V3_Large_320_FPN_Weights.DEFAULT
    reference_detector = fasterrcnn_mobilenet_v3_large_320_fpn(weights=weights).eval()
    sdk_summary.append({"path": "torchvision", "status": "loaded", "weights": str(weights)})
else:
    sdk_summary.append({"path": "torchvision", "status": "skipped", "reason": "networked official weights are optional"})

ULTRALYTICS_TESTED_VERSION = "8.4.138"
ENABLE_ULTRALYTICS = os.getenv("CV_ENABLE_ULTRALYTICS", "0") == "1"
if ENABLE_ULTRALYTICS:
    installed = importlib.metadata.version("ultralytics")
    if installed != ULTRALYTICS_TESTED_VERSION:
        raise RuntimeError(f"Expected ultralytics=={ULTRALYTICS_TESTED_VERSION}; found {installed}")
    from ultralytics import YOLO
    yolo_detector = YOLO("yolo26n.pt")
    sdk_summary.append({"path": "ultralytics", "status": "loaded", "version": installed, "checkpoint": "yolo26n.pt"})
else:
    sdk_summary.append({"path": "ultralytics", "status": "skipped", "license": "AGPL-3.0 or enterprise; not a default dependency"})

ENABLE_GROUNDING_DINO = os.getenv("CV_ENABLE_GROUNDING_DINO", "0") == "1"
GROUNDING_DINO_MODEL = "IDEA-Research/grounding-dino-tiny"
GROUNDING_DINO_REVISION = os.getenv("CV_GROUNDING_DINO_REVISION")
if ENABLE_GROUNDING_DINO:
    if not GROUNDING_DINO_REVISION:
        raise RuntimeError("Set CV_GROUNDING_DINO_REVISION to an reviewed immutable model commit")
    from transformers import AutoModelForZeroShotObjectDetection, AutoProcessor
    grounding_processor = AutoProcessor.from_pretrained(GROUNDING_DINO_MODEL, revision=GROUNDING_DINO_REVISION)
    grounding_detector = AutoModelForZeroShotObjectDetection.from_pretrained(GROUNDING_DINO_MODEL, revision=GROUNDING_DINO_REVISION)
    sdk_summary.append({"path": "transformers", "status": "loaded", "model": GROUNDING_DINO_MODEL, "revision": GROUNDING_DINO_REVISION})
else:
    sdk_summary.append({"path": "transformers", "status": "skipped", "reason": "requires an explicit immutable revision and prompt evaluation"})

pd.DataFrame(sdk_summary)

## 12. Closed-set, open-vocabulary category detection, and phrase grounding

![Closed-set detection uses a fixed category head; open-vocabulary detection aligns image regions with text prompts.](assets/closed-vs-open-vocabulary.svg)

These related contracts are not identical:

| Contract | Typical input | Typical question |
| --- | --- | --- |
| Open-vocabulary category detection | category names, including categories not fixed in the training taxonomy | “Where are the forklifts?” |
| Phrase grounding | a noun phrase or referring expression with attributes or relations | “the red fastener beside the housing” |

Grounding DINO-style systems can support both category-like prompts and richer phrase-conditioned localization. Prompt wording, synonyms, attributes, thresholding, and model revision must become versioned evidence. A phrase-conditioned box is not automatically correct merely because the requested noun appears in the prompt.

Course 09 will evaluate these contracts directly. Here the optional adapter establishes the integration boundary without turning unpublished local execution into a benchmark claim.

## 13. Save the evidence bundle and enterprise decision

The final choice is not “whichever model has the highest mAP.” It combines source/size quality, threshold sensitivity, latency, calibration, software/checkpoint license, export/runtime evidence, open-vocabulary need, privacy, and review workflow.

Only the two tiny dense runs have local model evidence. The other architecture options remain explicit evaluation gates.

In [ ]:
selected_run = final_evaluation.iloc[0].run
selected_metrics = final_evaluation.set_index("run").loc[selected_run].to_dict()
selected_sizes = source_size_evaluation[source_size_evaluation.run.eq(selected_run)].to_dict(orient="records")
selected_calibration = calibration_summary.set_index("run").loc[selected_run].to_dict() if selected_run in calibration_summary.run.values else {}

enterprise_options = pd.DataFrame([
    {
        "option": "TorchVision two-stage baseline",
        "local_evidence": "adapter not executed; maintained Faster R-CNN API reviewed",
        "best_fit": "mature proposal/ROI baseline and localization-focused comparison",
        "quality_gate": "evaluate AP50:95, small objects, duplicates, and source slices",
        "systems_gate": "profile preprocessing, proposals, ROI head, NMS, export, and target hardware",
        "license_governance": "review TorchVision code and official weight metadata",
        "status": "benchmark next",
    },
    {
        "option": "Dense anchor-free / YOLO-style path",
        "local_evidence": f"{selected_run}: AP50={selected_metrics['AP50']:.3f}, AP50:95={selected_metrics['AP50_95']:.3f}",
        "best_fit": "inspectable dense assignment and latency-oriented deployment",
        "quality_gate": "replace tiny model with maintained candidate under identical protocol",
        "systems_gate": "validate decode/NMS or end-to-end mode, export parity, memory, tails",
        "license_governance": "Ultralytics requires AGPL-3.0 compliance or enterprise terms",
        "status": "local teaching baseline",
    },
    {
        "option": "DETR-style set prediction",
        "local_evidence": "Hungarian cost and one-to-one assignment executed; no detector benchmark",
        "best_fit": "global matching and NMS-free set objective",
        "quality_gate": "benchmark maintained RT/Deformable/DINO-style checkpoint on same slices",
        "systems_gate": "measure query count, multi-scale attention, export, latency, memory",
        "license_governance": "pin implementation, model revision, weights, preprocessing, and license",
        "status": "candidate evaluation",
    },
    {
        "option": "Open-vocabulary extension",
        "local_evidence": "Grounding DINO adapter disabled; no local grounding claim",
        "best_fit": "dynamic categories or phrase grounding after closed-set need is insufficient",
        "quality_gate": "evaluate prompt variants, unknowns, false grounding, source and size slices",
        "systems_gate": "profile text+image encoding, prompt caching, thresholds, deployment runtime",
        "license_governance": "immutable model revision, checkpoint digest, model card, prompt audit",
        "status": "defer to Course 09 unless required",
    },
])
display(enterprise_options)

anchor_coverage.to_csv(ARTIFACT_DIR / "anchor_coverage.csv", index=False)
training_history.to_csv(ARTIFACT_DIR / "training_history.csv", index=False)
model_evaluation.to_csv(ARTIFACT_DIR / "model_evaluation.csv", index=False)
threshold_sweep.to_csv(ARTIFACT_DIR / "threshold_sweep.csv", index=False)
final_evaluation.to_csv(ARTIFACT_DIR / "final_evaluation.csv", index=False)
source_size_evaluation.to_csv(ARTIFACT_DIR / "source_size_evaluation.csv", index=False)
calibration_results.to_csv(ARTIFACT_DIR / "calibration.csv", index=False)
ap50_ap75_example.to_csv(ARTIFACT_DIR / "ap50_ap75_example.csv", index=False)
nms_crowding_example.to_csv(ARTIFACT_DIR / "nms_crowding_example.csv", index=False)
mitigation_comparison.to_csv(ARTIFACT_DIR / "mitigation_comparison.csv", index=False)
error_taxonomy_events.to_csv(ARTIFACT_DIR / "error_taxonomy_events.csv", index=False)
error_taxonomy_summary.to_csv(ARTIFACT_DIR / "error_taxonomy_summary.csv", index=False)
failure_slices.to_csv(ARTIFACT_DIR / "failure_slices.csv", index=False)
latency_results.to_csv(ARTIFACT_DIR / "latency.csv", index=False)
hungarian_assignment.to_csv(ARTIFACT_DIR / "hungarian_assignment.csv", index=False)
hungarian_cost_decomposition.to_csv(ARTIFACT_DIR / "hungarian_cost_decomposition.csv", index=False)
enterprise_options.to_csv(ARTIFACT_DIR / "enterprise_options.csv", index=False)
(ARTIFACT_DIR / "box_contract_examples.json").write_text(json.dumps(box_contract_examples, indent=2), encoding="utf-8")

detection_records = []
for prediction in final_predictions[selected_run]:
    for box, label, score in zip(prediction["boxes"], prediction["labels"], prediction["scores"]):
        detection_records.append({
            "sample_id": prediction["sample_id"],
            "box_xyxy": [float(value) for value in box],
            "label": CLASS_NAMES[int(label)],
            "score": float(score),
        })
(ARTIFACT_DIR / "held_out_detections.json").write_text(json.dumps(detection_records, indent=2), encoding="utf-8")

decision = {
    "course": "Beginner 05 — Object Detection",
    "scenario": "procedural multi-object assembly-line detection with Factory C held out",
    "selected_local_run": selected_run,
    "selection_rule": "highest held-out Factory C AP50 after choosing confidence and NMS on development A/B only",
    "threshold_notice": DEMONSTRATION_THRESHOLD_NOTICE,
    "selected_metrics": selected_metrics,
    "selected_size_slices": selected_sizes,
    "selected_calibration": selected_calibration,
    "failure_taxonomy": error_taxonomy_summary.to_dict(orient="records"),
    "mitigation_limits": mitigation_limits,
    "size_contract": {"small": "area < 1.25% image", "medium": "1.25% ≤ area < 4%", "large": "area ≥ 4%"},
    "options": enterprise_options.to_dict(orient="records"),
    "sdk_review": sdk_summary,
    "risk_boundaries": [
        "procedural data and a tiny detector do not certify a production system",
        "AP is computed by a transparent teaching evaluator, not official COCO API",
        "confidence is evaluated after thresholding and NMS and is not probability by default",
        "Factory C is held out from training and operating-point selection",
        "small-object and source slices can invalidate a strong aggregate score",
        "photometric augmentation addresses only tested capture factors and is not a general source-shift solution",
        "Ultralytics code/models require AGPL-3.0 compliance or applicable enterprise terms",
        "open-vocabulary prompts expand the validation surface and do not guarantee grounding",
    ],
    "production_next_steps": [
        "replace procedural images with governed, deduplicated, source/time-held-out data",
        "run official COCO-style evaluation plus decision-cost metrics",
        "benchmark one maintained two-stage, dense, and set-prediction checkpoint under one contract",
        "validate annotation agreement, small-object capture, crowding, and rare classes",
        "calibrate on a separate split and define class-specific abstention/review thresholds",
        "pin source, package, checkpoint digest, preprocessing, taxonomy, and license",
        "export through the intended runtime and profile complete tails on target hardware",
        "shadow deploy with monitoring, rollback, retention, privacy, and incident controls",
    ],
}
(ARTIFACT_DIR / "detector_decision.json").write_text(json.dumps(decision, indent=2), encoding="utf-8")
print(json.dumps(decision, indent=2))

## 14. What you should now be able to explain without code

1. Why is detection a variable-size structured prediction problem?
2. Why is `xywh` ambiguous unless center or top-left is named?
3. When does a correct-class prediction become a false positive?
4. Why does AP50 tolerate errors that AP50:95 exposes?
5. Why are small objects sensitive to stride and annotation error?
6. Where does IoU appear in training, NMS, and evaluation?
7. What do anchors provide, and what hyperparameters do they introduce?
8. Which assignment choices remain in an anchor-free detector?
9. Why does an FPN fuse shallow detail with deep semantics?
10. Why can focal loss help without fixing every imbalance?
11. Why can NMS suppress a real crowded-scene object?
12. Why are DETR object queries not text labels?
13. How does Hungarian matching support a one-to-one set objective?
14. Why is DINO detector not Course 04's DINO representation learner?
15. Why can confidence increase while reliability decreases after shift?
16. Why is the highest-AP candidate not automatically the deployment choice?
17. How do open-vocabulary category detection and phrase grounding differ?

## Exercises

- Add COCO top-left `xywh` round-trip assertions and reject ambiguous schemas.
- Implement Soft-NMS and compare crowded-scene recall with greedy NMS.
- Add a finer 16×16 feature head; measure small-object recall and latency.
- Change focal-loss gamma and inspect easy-negative contribution.
- Sweep Hungarian class/L1/GIoU weights and explain assignment changes.
- Replace the teaching AP with official `pycocotools` and document protocol differences.
- Enable exactly one maintained detector adapter, pin every artifact, and add it to the same source/size/latency evaluation.

## Production upgrade path

- governed annotations, taxonomy, provenance, ignore/crowd rules, and immutable split manifests;
- distributed training with reproducible assignment, augmentation, mixed precision, and recovery;
- official evaluator parity, confidence/localization calibration, and rare-event slices;
- exact runtime export, numerical parity, preprocessing/decode integration, and tail latency;
- versioned NMS/end-to-end duplicate policy and monitoring of pre/post candidate counts;
- identity, access, encryption, retention, privacy review, and human escalation;
- shadow deployment, canaries, rollback, audit logs, incident response, and model retirement.

**Next:** Course 06 replaces rectangular localization with semantic, instance, and promptable segmentation masks.